In [ ]:
import glob
import math
import os
import time

from datasets import load_dataset

# ══════════════════════════════════════════════════════════════════════════════
#  CẤU HÌNH
# ══════════════════════════════════════════════════════════════════════════════

# Glob pattern đến các file đầu vào
INPUT_PATTERN = "data/input/*.parquet"

# Thư mục chứa các file output (tạo mới nếu chưa có)
OUTPUT_DIR = "data/output"

# Số dòng tối đa mỗi file output.
# None → tự tính để số file output == số file input
ROWS_PER_OUTPUT_FILE = None

# Các cột dùng để xác định trùng lặp.
# None  → so sánh TẤT CẢ cột
# list  → vd: ["cleaned_text"]  hoặc  ["cleaned_text", "label"]
SUBSET_COLS = None

# Kích thước batch khi xử lý (tăng nếu RAM dư, giảm nếu bị OOM)
BATCH_SIZE = 50_000

# Thư mục cache HuggingFace — đổi nếu ổ mặc định (~/.cache) không đủ dung lượng
# os.environ["HF_DATASETS_CACHE"] = "/data/hf_cache"

# ══════════════════════════════════════════════════════════════════════════════


def make_dedup_fn(subset_cols):
    """Trả về hàm batch-map lọc trùng, dùng chung 1 seen-set (closure)."""
    seen = set()

    def dedup_batch(batch):
        cols = subset_cols if subset_cols else list(batch.keys())
        flags = []
        for row in zip(*[batch[c] for c in cols]):
            flags.append(row not in seen)
            seen.add(row)
        return {k: [v for v, keep in zip(batch[k], flags) if keep]
                for k in batch}

    return dedup_batch


def save_shards(ds_clean, output_dir, rows_per_file, n_input_files):
    """Chia dataset đã lọc thành nhiều file parquet."""
    n_clean = len(ds_clean)

    if rows_per_file is None:
        # giữ số file output ≈ số file input, chia đều
        rows_per_file = math.ceil(n_clean / n_input_files)

    n_shards = math.ceil(n_clean / rows_per_file)
    digits   = len(str(n_shards - 1))

    print(f"\n[3/3] Lưu {n_shards} file output "
          f"(≤ {rows_per_file:,} dòng/file) → {output_dir}/")

    total_bytes = 0
    for i in range(n_shards):
        start    = i * rows_per_file
        end      = min(start + rows_per_file, n_clean)
        shard    = ds_clean.select(range(start, end))
        out_path = os.path.join(output_dir, f"part-{str(i).zfill(digits)}.parquet")
        shard.to_parquet(out_path)
        size_kb   = os.path.getsize(out_path) / 1024
        total_bytes += os.path.getsize(out_path)
        print(f"    part-{str(i).zfill(digits)}.parquet  "
              f"{len(shard):>10,} dòng   {size_kb:>8,.0f} KB")

    print(f"\n  Tổng dung lượng output: {total_bytes/1024/1024:.1f} MB")
    return n_shards


def main():
    t_start = time.time()

    # ── 1. Tìm & load file đầu vào ──────────────────────────────────────────
    files = sorted(glob.glob(INPUT_PATTERN))
    if not files:
        raise FileNotFoundError(f"Không tìm thấy file nào khớp: {INPUT_PATTERN}")

    print(f"[1/3] Load {len(files)} file đầu vào:")
    for f in files:
        size_mb = os.path.getsize(f) / 1024 / 1024
        print(f"    {os.path.basename(f)}  ({size_mb:.1f} MB)")

    ds = load_dataset("parquet", data_files=files, split="train")
    n_original = len(ds)
    print(f"\n  Tổng dòng: {n_original:,}")
    print(f"  Cột      : {ds.column_names}")

    # ── 2. Dedup toàn cục ───────────────────────────────────────────────────
    subset_label = str(SUBSET_COLS) if SUBSET_COLS else "tất cả cột"
    print(f"\n[2/3] Dedup toàn cục (so sánh theo: {subset_label}) ...")

    dedup_fn = make_dedup_fn(SUBSET_COLS)
    ds_clean = ds.map(
        dedup_fn,
        batched=True,
        batch_size=BATCH_SIZE,
        desc="Deduplicating",
    )

    n_clean   = len(ds_clean)
    n_removed = n_original - n_clean
    print(f"\n  Dòng ban đầu   : {n_original:>12,}")
    print(f"  Dòng trùng xoá : {n_removed:>12,}  ({n_removed/n_original*100:.2f}%)")
    print(f"  Dòng còn lại   : {n_clean:>12,}")

    # ── 3. Lưu output ───────────────────────────────────────────────────────
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    n_out = save_shards(ds_clean, OUTPUT_DIR, ROWS_PER_OUTPUT_FILE, len(files))

    elapsed = time.time() - t_start
    print(f"\n✅  Xong!  {len(files)} file → {n_out} file   "
          f"({n_original:,} → {n_clean:,} dòng)   "
          f"Thời gian: {elapsed:.1f}s")


if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import zipfile
import time
from pathlib import Path


# ══════════════════════════════════════════════════════════════════════════════
#  PUBLIC API
# ══════════════════════════════════════════════════════════════════════════════

def zip_folder(
    folder: str,
    output_zip: str | None = None,
    pattern: str = "*.parquet",
    compression: int = zipfile.ZIP_DEFLATED,
    compresslevel: int = 6,
) -> str:
    """
    Nén tất cả file khớp `pattern` trong `folder` thành 1 file ZIP.

    Tham số:
        folder       : Thư mục chứa file cần nén.
        output_zip   : Đường dẫn file ZIP đầu ra.
                       Mặc định: <folder>.zip (cùng cấp với folder).
        pattern      : Glob pattern lọc file, mặc định "*.parquet".
        compression  : Thuật toán nén.
                         ZIP_DEFLATED (mặc định) – cân bằng tốc độ/dung lượng.
                         ZIP_BZIP2               – nén mạnh hơn, chậm hơn.
                         ZIP_LZMA                – nén mạnh nhất, chậm nhất.
        compresslevel: Mức nén 1–9 (chỉ áp dụng với DEFLATED và BZIP2).

    Trả về:
        Đường dẫn file ZIP đã tạo.
    """
    folder = str(folder)
    files  = sorted(glob.glob(os.path.join(folder, pattern)))

    if not files:
        raise FileNotFoundError(
            f"Không tìm thấy file nào khớp '{pattern}' trong: {folder}"
        )

    if output_zip is None:
        output_zip = folder.rstrip("/\\") + ".zip"

    return _write_zip(files, output_zip, folder, compression, compresslevel)


def zip_files(
    files: list[str],
    output_zip: str,
    arcname_prefix: str = "",
    compression: int = zipfile.ZIP_DEFLATED,
    compresslevel: int = 6,
) -> str:
    """
    Nén danh sách file tuỳ ý thành 1 file ZIP.

    Tham số:
        files          : Danh sách đường dẫn tuyệt đối / tương đối.
        output_zip     : Đường dẫn file ZIP đầu ra.
        arcname_prefix : Tiền tố thư mục bên trong ZIP.
                         Ví dụ: "output/" → các file nằm trong thư mục output/ trong ZIP.
        compression    : Xem zip_folder.
        compresslevel  : Xem zip_folder.

    Trả về:
        Đường dẫn file ZIP đã tạo.
    """
    if not files:
        raise ValueError("Danh sách file trống.")

    base_dir = os.path.commonpath([os.path.abspath(f) for f in files])
    return _write_zip(files, output_zip, base_dir, compression, compresslevel,
                      arcname_prefix=arcname_prefix)


# ══════════════════════════════════════════════════════════════════════════════
#  INTERNAL
# ══════════════════════════════════════════════════════════════════════════════

def _write_zip(
    files: list[str],
    output_zip: str,
    base_dir: str,
    compression: int,
    compresslevel: int,
    arcname_prefix: str = "",
) -> str:
    os.makedirs(os.path.dirname(os.path.abspath(output_zip)), exist_ok=True)

    total_original = sum(os.path.getsize(f) for f in files)
    t0 = time.time()

    print(f"Nén {len(files)} file → {output_zip}")
    print(f"  Dung lượng gốc : {_fmt_size(total_original)}")

    with zipfile.ZipFile(output_zip, "w", compression=compression,
                         compresslevel=compresslevel) as zf:
        for f in files:
            arcname = arcname_prefix + os.path.relpath(f, base_dir)
            zf.write(f, arcname=arcname)
            size_orig = os.path.getsize(f)
            info      = zf.getinfo(arcname)
            print(f"    + {arcname:<35} "
                  f"{_fmt_size(size_orig):>9} → {_fmt_size(info.compress_size):>9}")

    zip_size  = os.path.getsize(output_zip)
    ratio     = (1 - zip_size / total_original) * 100 if total_original else 0
    elapsed   = time.time() - t0

    print(f"\n  Dung lượng ZIP  : {_fmt_size(zip_size)}  "
          f"(giảm {ratio:.1f}%,  {elapsed:.1f}s)")
    return output_zip


def _fmt_size(n_bytes: int) -> str:
    for unit in ("B", "KB", "MB", "GB"):
        if n_bytes < 1024:
            return f"{n_bytes:.1f} {unit}"
        n_bytes /= 1024
    return f"{n_bytes:.1f} TB"


# ══════════════════════════════════════════════════════════════════════════════
#  CHẠY ĐỘC LẬP
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="Nén thư mục Parquet thành ZIP")
    parser.add_argument("folder",      help="Thư mục chứa file cần nén")
    parser.add_argument("-o", "--out", help="Đường dẫn file ZIP đầu ra", default=None)
    parser.add_argument("-p", "--pattern", default="*.parquet",
                        help="Glob pattern (mặc định: *.parquet)")
    parser.add_argument("--algo", choices=["deflate", "bzip2", "lzma"],
                        default="deflate", help="Thuật toán nén")
    parser.add_argument("-l", "--level", type=int, default=6,
                        help="Mức nén 1-9 (mặc định: 6)")
    args = parser.parse_args()

    algo_map = {
        "deflate": zipfile.ZIP_DEFLATED,
        "bzip2":   zipfile.ZIP_BZIP2,
        "lzma":    zipfile.ZIP_LZMA,
    }

    zip_folder(
        folder       = args.folder,
        output_zip   = args.out,
        pattern      = args.pattern,
        compression  = algo_map[args.algo],
        compresslevel= args.level,
    )
